# 07. Data Encoding and Splitting

Encode categorical variables and split data for machine learning.

## Objectives
- Load standardized features from previous step
- Analyze and encode categorical data (multi-hot and one-hot)
- Split data into train/test/validation sets

## Table of Contents
1. [Load Data](#load-data)
2. [Event Categories Analysis & Feature Encoding](#event-categories-analysis--feature-encoding)
   - 2.1 [Event Categories Discovery](#event-categories-discovery)
   - 2.2 [Multi-Hot Encoding for Events](#multi-hot-encoding-for-events)
   - 2.3 [One-Hot Encoding for Categorical Variables](#one-hot-encoding-for-categorical-variables)
3. [Data Splitting](#data-splitting)

---

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [ ]:
# Define paths
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
OUT_DIR = PROCESSED_DIR

# Load derived features
data = pd.read_csv(PROCESSED_DIR / "features_derived.csv", sep=None, engine="python", encoding="utf-8-sig")
print(f"Data shape: {data.shape}")
data.head()

## 2. Event Categories Analysis & Feature Encoding

Analyze orientation events data and apply encoding techniques for all categorical variables.

### 2.1 Event Categories Discovery

Explore and clean the orientation events data to understand what categories exist.

In [ ]:
# Clean and analyze event categories
event_series = (
    data['sdo2_orientation_Event_Types_Attended']
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace("online opleidingspresentatie", "online_opleidingspresentatie", regex=False)
    .str.replace("open dag", "open_dag", regex=False)
    .str.split(r'\s*,\s*')
    .explode()
    .str.strip()
)

print("Unique events:", sorted(event_series.unique()))
print("Event counts:\n", event_series.value_counts())

### 2.2 Multi-Hot Encoding for Events

Convert comma-separated event values into binary indicator columns.

In [ ]:
# Multi-hot encoding for event categories
expected_events = sorted(event_series.unique())
expected_cols = [(e if e == "absent" else f"attended_{e}") for e in expected_events]

if all(col in data.columns for col in expected_cols):
    print("✓ Multi-hot encoding already done. Skipping.")
else:
    # Create multi-hot matrix
    tmp = event_series.reset_index()
    tmp.columns = ["row_id", "event"]
    multi_hot = pd.crosstab(tmp["row_id"], tmp["event"]).astype("Int64")
    
    # Rename columns: "absent" stays as-is, others get "attended_" prefix
    rename_map = {c: (c if c == "absent" else f"attended_{c}") for c in multi_hot.columns}
    multi_hot.rename(columns=rename_map, inplace=True)
    
    # Join and fill missing values
    data = data.join(multi_hot, how="left")
    data[list(multi_hot.columns)] = data[list(multi_hot.columns)].fillna(0).astype("Int64")
    print("✓ Created event columns:", list(multi_hot.columns))

# Remove original column
data.drop(columns=["sdo2_orientation_Event_Types_Attended"], inplace=True, errors="ignore")

### 2.3 One-Hot Encoding for Categorical Variables

Apply standard one-hot encoding to remaining categorical columns.

In [ ]:
# One-hot encode remaining categorical columns
encode_cols = [
    'sdo5_degree_degree',
    'sdo1_previous_Previous_Education_Type', 
    'sdo2_skc_ADVIES_DEF',
    'age_category'
]

data = pd.get_dummies(data, columns=encode_cols, drop_first=False, dtype='int')
print(f"Final data shape: {data.shape}")

## 3. Data Splitting

Data will be split using a hybrid-temporal approach:
- Training: 2021 and 55% of 2022
- Validation: 45% of 2022
- Test: 2023

In [ ]:
# Assign data splits based on year
data['set'] = np.where(data['sdo5_degree_COLLEGEJAAR'] == 2021, 'train',
                np.where(data['sdo5_degree_COLLEGEJAAR'] == 2023, 'test', 'val'))
random_2022 = data[data['sdo5_degree_COLLEGEJAAR'] == 2022].sample(frac=0.55, random_state=42).index
data.loc[random_2022, 'set'] = 'train'
data.loc[(data['sdo5_degree_COLLEGEJAAR'] == 2022) & (~data.index.isin(random_2022)), 'set'] = 'val'

# Print split counts
print("Data split counts:\n", data['set'].value_counts())

## 4. Save dataset

In [ ]:
# Save dataset
output_path = OUT_DIR / "encoded_and_split_data.csv"
data.to_csv(output_path, index=False)

print("✅ Data encoding and splitting complete!")
print(f"   📂 Saved: {output_path}")
print(f"   📊 Shape: {data.shape}")
print(f"   🏷️ Columns: {len(data.columns)}")

# Show data type summary
print(f"\n📋 Data type summary:")
dtype_counts = data.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"   {dtype}: {count} columns")